
# 06 - What the encoding costs

**No training. A few minutes on CPU.** Produces Figure 3 of the paper.

The proposal claims the traversal encoding is cheap: one breadth-first traversal per node,
$O(|V|+|E|)$ each. That claim needs numbers rather than a complexity class, for two reasons.

First, the honest comparison is against the *alternatives*. Laplacian PE needs an
eigendecomposition and random-walk PE needs repeated sparse products, so "cheap" only means
something relative to those.

Second, and more importantly, **preprocessing time is not the real cost of our method** -
the rewiring is. Adding every pair within $r$ hops multiplies the edge count, and that
multiplier lands on every message-passing step of every epoch. A method that preprocesses in
one second and then makes training seven times slower is not a cheap method, and reporting
only the preprocessing time would be misleading. Both are measured here.

In [ ]:
# Colab: torch ships preinstalled, torch-geometric does not.
!pip -q install torch-geometric

In [ ]:

import os, time, math, json
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch_geometric.datasets import GNNBenchmarkDataset
from torch_geometric.transforms import AddLaplacianEigenvectorPE, AddRandomWalkPE

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    RESULT_ROOT, DATA_ROOT = "/content/drive/MyDrive/trpe", "/content/trpe_data"
except Exception:
    RESULT_ROOT, DATA_ROOT = os.path.abspath("./trpe"), os.path.abspath("./trpe_data")
OUT_DIR = os.path.join(RESULT_ROOT, "results", "efficiency")
FIG_DIR = os.path.join(RESULT_ROOT, "figures")
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)

LAP_K, RW_STEPS, N_SAMPLE = 8, 16, 120

In [ ]:

def trpe(data, r):
    n = int(data.num_nodes); ei = data.edge_index.numpy()
    A = np.zeros((n, n)); A[ei[0], ei[1]] = 1.0
    A = np.maximum(A, A.T); np.fill_diagonal(A, 0.0)
    dist = np.full((n, n), -1, dtype=np.int64); np.fill_diagonal(dist, 0)
    Mk = np.eye(n)
    for k in range(1, r + 1):
        Mk = Mk @ A
        fresh = (Mk > 0.5) & (dist < 0)
        if not fresh.any(): break
        dist[fresh] = k
    return int((dist >= 1).sum())


def timeit(fn, graphs):
    t0 = time.perf_counter()
    out = [fn(g) for g in graphs]
    return (time.perf_counter() - t0) / len(graphs) * 1000, out   # ms per graph


rows = []
for name, split in [("CSL", None), ("CLUSTER", "val"), ("CIFAR10", "val")]:
    ds = (GNNBenchmarkDataset(os.path.join(DATA_ROOT, name), name, split=split) if split
          else GNNBenchmarkDataset(os.path.join(DATA_ROOT, name), name))
    idx = np.linspace(0, len(ds) - 1, min(N_SAMPLE, len(ds))).astype(int)
    graphs = [ds[int(i)] for i in idx]
    base_edges = float(np.mean([g.edge_index.size(1) for g in graphs]))
    n_nodes = float(np.mean([g.num_nodes for g in graphs]))

    row = dict(dataset=name, avg_nodes=n_nodes, base_edges=base_edges)
    for r in (2, 3):
        ms, counts = timeit(lambda g: trpe(g, r), graphs)
        row[f"trpe_r{r}_ms"] = ms
        row[f"trpe_r{r}_edges"] = float(np.mean(counts))
        row[f"blowup_r{r}"] = float(np.mean(counts)) / base_edges

    def lap(g):
        k = min(LAP_K, max(1, int(g.num_nodes) - 2))
        try:    return AddLaplacianEigenvectorPE(k=k, attr_name="pe", is_undirected=True)(g.clone())
        except Exception: return None
    def rw(g):
        try:    return AddRandomWalkPE(walk_length=RW_STEPS, attr_name="pe")(g.clone())
        except Exception: return None

    row["lappe_ms"], _ = timeit(lap, graphs)
    row["rwpe_ms"],  _ = timeit(rw,  graphs)
    rows.append(row)

hdr = (f"{'dataset':<10}{'avg n':>7}{'|E|':>9}{'T-RPE r2':>10}{'T-RPE r3':>10}"
       f"{'LapPE':>9}{'RWPE':>8}   {'|E_2|/|E|':>10}{'|E_3|/|E|':>10}")
print(hdr); print("-" * len(hdr))
print(f"{'':<10}{'':>7}{'':>9}{'ms/graph':>10}{'ms/graph':>10}{'ms/graph':>9}{'ms/graph':>8}")
for r in rows:
    print(f"{r['dataset']:<10}{r['avg_nodes']:>7.0f}{r['base_edges']:>9.0f}"
          f"{r['trpe_r2_ms']:>10.2f}{r['trpe_r3_ms']:>10.2f}{r['lappe_ms']:>9.2f}{r['rwpe_ms']:>8.2f}"
          f"{r['blowup_r2']:>10.1f}x{r['blowup_r3']:>9.1f}x")

json.dump(rows, open(os.path.join(OUT_DIR, "efficiency.json"), "w"), indent=2)
print("\nwritten ->", os.path.join(OUT_DIR, "efficiency.json"))

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
names = [r["dataset"] for r in rows]
xs = np.arange(len(names)); w = 0.2

for i, (key, lab, col) in enumerate([("trpe_r2_ms", "T-RPE $r{=}2$", "#3b6ea5"),
                                     ("trpe_r3_ms", "T-RPE $r{=}3$", "#7fa8d0"),
                                     ("lappe_ms",   "LapPE",         "#c76a4a"),
                                     ("rwpe_ms",    "RWPE",          "#d9a441")]):
    axes[0].bar(xs + (i - 1.5) * w, [r[key] for r in rows], w, label=lab, color=col)
axes[0].set_xticks(xs); axes[0].set_xticklabels(names)
axes[0].set_ylabel("preprocessing, ms / graph"); axes[0].set_yscale("log")
axes[0].set_title("One-off encoding cost"); axes[0].legend(frameon=False, fontsize=8)

for i, (key, lab, col) in enumerate([("blowup_r2", "$r{=}2$", "#3b6ea5"),
                                     ("blowup_r3", "$r{=}3$", "#7fa8d0")]):
    axes[1].bar(xs + (i - 0.5) * w * 2, [r[key] for r in rows], w * 2, label=lab, color=col)
axes[1].axhline(1.0, color="k", lw=0.8, ls="--")
axes[1].set_xticks(xs); axes[1].set_xticklabels(names)
axes[1].set_ylabel(r"$|E_r| / |E|$")
axes[1].set_title("Rewiring blow-up = per-epoch training multiplier")
axes[1].legend(frameon=False, fontsize=8)
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig_efficiency.pdf"), bbox_inches="tight")
plt.show()
print("saved ->", os.path.join(FIG_DIR, "fig_efficiency.pdf"))


The right-hand panel is the one that constrains the project. The left panel is a one-off cost
paid once per dataset; the right panel is a multiplier on every epoch of every run, and it is
what makes the M3/M4 notebooks the slow ones in the grid. When the paper claims the encoding
is cheap, it should claim it about preprocessing and state the rewiring cost in the same
breath.